In [1]:
import os
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"
# Set JAVA_HOME and update PATH for this session
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Importing required Spark libraries 

from pyspark.sql import SparkSession
from pyspark.sql.functions import mean, when, col

# Initializing a Spark session
spark = SparkSession.builder.appName("DiabetesAnalysis").getOrCreate()

# Loading the CSV file into a DataFrame
df = spark.read.csv("diabetes.csv", header=True, inferSchema=True)

# Displaying the schema to verify column names and data types
print("DataFrame Schema:")
df.printSchema()

# Showing the first few rows of the original DataFrame
print("\nOriginal DataFrame (first 5 rows):")
df.show(5)

# Calculating mean BMI, excluding zero values
mean_bmi = df.filter(df.BMI > 0).select(mean("BMI")).collect()[0][0]
print(f"\nMean BMI (excluding zeros): {mean_bmi:.2f}")

# Creating new DataFrame replacing BMI=0 with mean BMI
pyspark_df_nonzero_BMI = df.withColumn("BMI", when(df.BMI == 0, mean_bmi).otherwise(df.BMI))

# Showing the first few rows of the updated DataFrame
print("\npyspark_df_nonzero_BMI (first 5 rows):")
pyspark_df_nonzero_BMI.show(5)

# Creating a new dataframe pyspark_df_outcome that shows rows with age >= 35
# and Diabetes Pedigree Function >= 0.51
pyspark_df_outcome = pyspark_df_nonzero_BMI.filter(
    (col("Age") >= 35) & 
    (col("DiabetesPedigreeFunction") >= 0.51)
)

# Showing the filtered DataFrame
print("\npyspark_df_outcome (all rows meeting criteria):")
pyspark_df_outcome.show()

# Displaying summary statistics of the final DataFrame
print("\nSummary statistics of pyspark_df_outcome:")
pyspark_df_outcome.describe().show()

# Counting the number of rows in the final DataFrame
row_count = pyspark_df_outcome.count()
print(f"\nNumber of rows in pyspark_df_outcome: {row_count}")

# Stoping the Spark session
spark.stop()



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/27 20:06:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/10/27 20:06:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/10/27 20:06:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


DataFrame Schema:
root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)


Original DataFrame (first 5 rows):
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          6|    148|           72|           35|      0|33.6|                   0.627| 50|      1|
|          1|     85|           66|           29|      0|26.6|                   0.351| 31|      0|
|          8|    183|           64|            0|      0|

24/10/27 20:06:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------------+-----------------+------------------+
|summary|       Pregnancies|           Glucose|     BloodPressure|     SkinThickness|           Insulin|              BMI|DiabetesPedigreeFunction|              Age|           Outcome|
+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------------+-----------------+------------------+
|  count|               101|               101|               101|               101|               101|              101|                     101|              101|               101|
|   mean|6.5049504950495045|132.02970297029702|  74.0990099009901|22.821782178217823| 95.94059405940594| 33.3094798383405|      0.8089009900990101|47.78217821782178|0.5841584158415841|
| stddev| 3.645884700251059| 35.88549998134192|18.536183507127372|17.138492